# Imports

In [1]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from scipy.special import expit

from sklearn.frozen import FrozenEstimator
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split, cross_val_predict

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [3]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)


def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)


def voting_cross_val_predict(probas: list[np.ndarray], weights: np.ndarray[float]) -> np.ndarray[float]:
    weights /= weights.sum()
    return np.sum([proba * weight for proba, weight in zip(probas, weights)], axis=0)

# Loading Dataset

In [4]:
X_train_proba = pd.read_parquet('../data/processed/X_train_stacking.parquet')
X_test_proba = pd.read_parquet('../data/processed/X_test_stacking.parquet')

In [5]:
X_train = pd.read_parquet('../data/processed/X_train.parquet')
X_test = pd.read_parquet('../data/processed/X_test.parquet')

y_train = pd.read_parquet('../data/processed/y_train.parquet')

In [6]:
X_train_proba.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043


In [7]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Feature Engineering

## Ridge

In [8]:
ridge = load_pickle('../models/model_ridge.pkl')

In [9]:
X_train_proba['ridge_logit'] = cross_val_predict(ridge, X_train, y_train.PitNextLap,  cv=StratifiedKFold(shuffle=True), n_jobs=-1, method='decision_function')
X_test_proba['ridge_logit'] = ridge.decision_function(X_test)

## Voting Classifier

In [10]:
X_train_proba['voting_lgbm_cat_xgb_hist_proba'] = voting_cross_val_predict(
    [
        X_train_proba['lgbm_proba'],
        X_train_proba['cat_proba'],
        X_train_proba['xgb_proba'],
        X_train_proba['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [11]:
X_train_proba['voting_lg_ridge_proba'] = voting_cross_val_predict(
    [
        X_train_proba['lg_proba'],
        X_train_proba['ridge_logit'].apply(expit),
    ],
    weights=np.power([0.8934924863693583, 0.8880861768599859], 2)
)

In [12]:
X_test_proba['voting_lgbm_cat_xgb_hist_proba'] = voting_cross_val_predict(
    [
        X_test_proba['lgbm_proba'],
        X_test_proba['cat_proba'],
        X_test_proba['xgb_proba'],
        X_test_proba['hist_proba']
    ], 
    weights=np.power([0.9484150827066593, 0.9481909576979181, 0.9477770044293032, 0.9485545192992528], 2)
)

In [14]:
X_test_proba['voting_lg_ridge_proba'] = voting_cross_val_predict(
    [
        X_test_proba['lg_proba'],
        X_test_proba['ridge_logit'],
    ], 
    weights=np.power([0.8934924863693583, 0.8880861768599859], 2)
)

# Machine Learning

###### 

In [15]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "objective": "binary",
        "n_estimators": trial.suggest_int("n_estimators", 30, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.003, 0.05, log=True),
        "max_depth": trial.suggest_int("max_depth", 1, 4),
        "num_leaves": trial.suggest_int("num_leaves", 2, 16),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 100, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 100, log=True),
        "verbosity": -1,
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

features_to_use = ['voting_lg_ridge_proba', 'knn_proba', 'voting_lgbm_cat_xgb_hist_proba']

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train_proba[features_to_use], y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-19 12:45:39,836] A new study created in memory with name: no-name-63a7376b-b1d7-4155-8b41-66465b223f0e
Best trial: 3. Best value: 0.946484:   0%|▏                                                                                                               | 1/500 [00:08<1:14:28,  8.95s/it]

[I 2026-05-19 12:45:48,768] Trial 3 finished with value: 0.9464844234063283 and parameters: {'n_estimators': 85, 'learning_rate': 0.04851550912367974, 'max_depth': 1, 'num_leaves': 3, 'min_child_samples': 63, 'subsample': 0.6292609612335408, 'colsample_bytree': 0.6727145721178005, 'reg_alpha': 4.9444203185243536e-05, 'reg_lambda': 4.463207248916472}. Best is trial 3 with value: 0.9464844234063283.


Best trial: 9. Best value: 0.946977:   1%|▉                                                                                                                 | 4/500 [00:09<12:40,  1.53s/it]

[I 2026-05-19 12:45:49,251] Trial 4 finished with value: 0.9408941025320459 and parameters: {'n_estimators': 96, 'learning_rate': 0.01730313599524915, 'max_depth': 1, 'num_leaves': 4, 'min_child_samples': 192, 'subsample': 0.9903163452610904, 'colsample_bytree': 0.6528573220582258, 'reg_alpha': 0.029972111060422007, 'reg_lambda': 0.15099048361604217}. Best is trial 3 with value: 0.9464844234063283.
[I 2026-05-19 12:45:49,348] Trial 0 finished with value: 0.9438135178695912 and parameters: {'n_estimators': 64, 'learning_rate': 0.012605723444627044, 'max_depth': 2, 'num_leaves': 16, 'min_child_samples': 151, 'subsample': 0.9376317719808214, 'colsample_bytree': 0.752366659695173, 'reg_alpha': 24.71610552599812, 'reg_lambda': 1.2588735460810603}. Best is trial 3 with value: 0.9464844234063283.
[I 2026-05-19 12:45:49,427] Trial 9 finished with value: 0.9469769567958007 and parameters: {'n_estimators': 56, 'learning_rate': 0.03305384650485767, 'max_depth': 4, 'num_leaves': 7, 'min_child_samp

Best trial: 9. Best value: 0.946977:   1%|█▏                                                                                                                | 5/500 [00:11<13:13,  1.60s/it]

...

[I 2026-05-19 13:10:54,400] Trial 498 finished with value: 0.9491925595290418 and parameters: {'n_estimators': 273, 'learning_rate': 0.00998917695223314, 'max_depth': 4, 'num_leaves': 5, 'min_child_samples': 240, 'subsample': 0.8752383644154257, 'colsample_bytree': 0.9775455450251381, 'reg_alpha': 0.044277078837069606, 'reg_lambda': 25.292524583006696}. Best is trial 492 with value: 0.9497476583824082.


Best trial: 492. Best value: 0.949748: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████▊| 499/500 [25:15<00:01,  1.02s/it]

[I 2026-05-19 13:10:55,035] Trial 497 finished with value: 0.9494266381249286 and parameters: {'n_estimators': 292, 'learning_rate': 0.009803871954013351, 'max_depth': 4, 'num_leaves': 5, 'min_child_samples': 204, 'subsample': 0.8790810759283725, 'colsample_bytree': 0.983425707353601, 'reg_alpha': 0.05341511231495445, 'reg_lambda': 3.6516958546913707}. Best is trial 492 with value: 0.9497476583824082.


Best trial: 492. Best value: 0.949748: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 500/500 [25:15<00:00,  3.03s/it]

[I 2026-05-19 13:10:55,475] Trial 499 finished with value: 0.9497407813150088 and parameters: {'n_estimators': 274, 'learning_rate': 0.047734855947167094, 'max_depth': 4, 'num_leaves': 5, 'min_child_samples': 220, 'subsample': 0.9277019288071781, 'colsample_bytree': 0.9786859166495131, 'reg_alpha': 0.042875462839447036, 'reg_lambda': 26.085384739854828}. Best is trial 492 with value: 0.9497476583824082.
Best trial score:
0.9497476583824082

Best params:
{'n_estimators': 293, 'learning_rate': 0.047663767796583004, 'max_depth': 4, 'num_leaves': 5, 'min_child_samples': 196, 'subsample': 0.8952983695946886, 'colsample_bytree': 0.982244251020764, 'reg_alpha': 0.06333722350987524, 'reg_lambda': 24.555536711114392}


## Train Dataset

In [16]:
cv = StratifiedKFold(shuffle=True, random_state=42, n_splits=5)

stacking_proba = cross_val_predict(
    LGBMClassifier(**study.best_params), 
    X_train_proba[features_to_use], 
    y_train.PitNextLap,
    cv=cv, 
    n_jobs=-1, 
    method='predict_proba'
)[:, 1]

In [17]:
X_train_stacking = X_train_proba.copy()
X_train_stacking['stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba'] = stacking_proba

# Test Dataset

In [18]:
stacking = LGBMClassifier(**study.best_params).fit(X_train_proba[features_to_use], y_train.PitNextLap)

[LightGBM] [Info] Number of positive: 87381, number of negative: 351759
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001116 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 439140, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198982 -> initscore=-1.392668
[LightGBM] [Info] Start training from score -1.392668


In [19]:
X_test_stacking = X_test_proba.copy()
X_test_stacking['stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba'] = stacking.predict_proba(X_test_proba[features_to_use])[:, 1]

# Saving

In [20]:
X_test_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_logit,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,-1.272709,0.019849,-0.629264,0.006334
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,-0.626960,0.012173,-0.279447,0.003675
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,-1.335019,0.010304,-0.661186,0.003484
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.190133,0.444558,0.424415,0.238053
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.535153,0.939648,0.688728,0.895339


In [21]:
X_train_stacking.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,ridge_logit,voting_lgbm_cat_xgb_hist_proba,voting_lg_ridge_proba,stacking_lgbm_with_knn_and_voting_lg_ridge_and_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.500714,0.868510,0.716126,0.773266
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.335105,0.772514,0.636769,0.591885
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.136991,0.717012,0.546431,0.516648
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,-1.247050,0.005494,0.112073,0.001113
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,-0.094611,0.482416,0.445065,0.293469


In [22]:
X_train_stacking.to_parquet('../data/processed/X_train_stacking2.parquet')
X_test_stacking.to_parquet('../data/processed/X_test_stacking2.parquet')

[LightGBM] [Info] Number of positive: 69905, number of negative: 281407
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.060958 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 351312, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392665
[LightGBM] [Info] Start training from score -1.392665
[LightGBM] [Info] Number of positive: 69905, number of negative: 281407
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062352 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 351312, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.198983 -> initscore=-1.392665
[LightGBM] [Info] Start training from score -1.392665
[LightGBM] [